# Week 3 Live Coding
## Reproduce the +0.7 pp ATE, and ask whether it could have happened by chance

The data file is a balanced subset of the actual public replication dataset from:

> Gerber, Huber, Fang & Gooch (2017). *The Generalizability of Social Pressure Effects on Turnout Across High-Salience Electoral Contexts: Field Experimental Evidence From 1.96 Million Citizens in 17 States.* American Politics Research 45(4): 533–559.

**About the subset.** The full replication archive has 1,969,899 voters in a 93%-treated / 7%-control split. To keep the file small and quick to load, we have taken a balanced sample (4,000 control + 4,000 treated per state, where available). The sample preserves the state mix and gives a clean unweighted average treatment effect very close to the paper's published number. The file has about 108,000 voters and is about 1.6 MB.

Columns:
- `state`: two-letter state code
- `treatment`: 1 if assigned to receive the report card mailer, 0 if control
- `voted_2014`: 1 if this voter actually voted in November 2014, 0 if not. **The outcome.**
- `vote_history_stratum`: `'below'`, `'average'`, or `'above'` the state median past-vote count
- `high_salience_state`: 1 if the state's 2014 election was high-salience, 0 if not. The paper scores each state 1 to 4 by adding up four things: a contested Senate race, a contested governor's race, and whether the Cook Political Report rated each of those a toss-up. States scoring 3 or 4 are coded 1 here. (Nine of the seventeen: AK, AR, CO, GA, IA, KS, ME, MI, NH.)

Three things we will do:
1. **Compute the ATE.** Reproduce the +0.7 percentage point average treatment effect, as a `groupby` and as a regression coefficient, and see why randomization makes it *causal*.
2. **Ask the data whether it's real.** Use *randomization inference*. If the mailer really did nothing, what would gaps like this look like just by chance? We generate the answer, then compare.
3. **Ask the data where the effect lives.** Subgroup by past-vote stratum and by state salience, and see why the 2008 paper reported a much larger number.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

Run the cell below to load the data.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

## Part 1: The two-line ATE

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/'
                 'data_science_campaigns_26/main/weeks/wk03_rcts_and_ate/data/gotv_2014.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df['treatment'].value_counts()

53,775 in each group. (The full paper has 1.97M voters in a 93/7 imbalance. We balanced the subset to keep the file small. The point estimate is essentially the same.)


### First, did the coin flip work?

Before trusting the comparison, check whether the two groups really do
look alike on what we can observe. This is the balance table from
the slides. The two columns should match to within a few tenths of a
percentage point.


In [ ]:
# normalize='columns' turns the counts into shares within each arm.
balance = pd.crosstab(df['vote_history_stratum'], df['treatment'],
                      normalize='columns')
print(round(100 * balance, 1))


Close, in every row. Nobody arranged that; a random number generator did.

And remember the limit from the slides: this table can only cover what we
observe. Nobody measured how much these voters cared about the 2014
election, and that is exactly the kind of thing a coin flip handles and a
control variable cannot.

### The computation

For each voter we have a treatment status (`treatment`, 0 or 1) and an outcome (`voted_2014`, 0 or 1). The average treatment effect is just:

`mean(voted_2014 | treated) − mean(voted_2014 | control)`

In pandas:

In [ ]:
df.groupby('treatment')['voted_2014'].mean()

In [ ]:
turnout_by_arm = df.groupby('treatment')['voted_2014'].mean()
control_mean = turnout_by_arm.loc[0]
treated_mean = turnout_by_arm.loc[1]
observed_ate = treated_mean - control_mean

print('Control group turnout:  ', round(100*control_mean, 3), '%')
print('Treatment group turnout:', round(100*treated_mean, 3), '%')
print('Observed ATE:            ', round(100*observed_ate, 3), 'percentage points')

**Look at what you just did.**

You just reproduced the headline result of a paper in *American Politics Research*! That `groupby` is the entire causal computation. It is a valid answer to "what did the mailer do" because of the coin flip. Every fancy statistical method you will ever see for an experiment is a refinement on these three lines.

**Now put it next to the donor's number.** The 2008 paper reports **+4.8 percentage points**. The 2017 replication finds **+0.7**. Both were randomized experiments. Both are right about the population they studied. The 2014 number is one-seventh the size of the donor's number, and it comes out of the same report card, sent in a midterm general election instead of a primary.

### The same ATE, as a regression coefficient

Last week you learned that a coefficient on a 0/1 predictor is just the difference in group averages. So this ATE is also a regression coefficient. Run it and compare to the `groupby` above.

In [ ]:
smf.ols('voted_2014 ~ treatment', data=df).fit().params

The `treatment` coefficient is about **0.0073**. A coefficient on a 0/1 variable is a proportion, so multiplying by 100 gives **+0.73 percentage points**, the same ATE the `groupby` gave us.

But something changed since last week. In Week 2 that kind of coefficient was only an *association*: the program districts differed from the others in hidden ways, so the gap was contaminated. Here, treatment was assigned by a **coin flip**, so the very same arithmetic (difference in group means) is now a **causal effect** (for the voters actually assigned and measured; there's some fine print we get to in Week 7). Randomization is what upgrades "difference in means" into "average treatment effect."

### What if we "control for" vote history, the way an observational study would?

In Week 2, adding a control column moved the program coefficient a lot (8.7 → 3.4 points). Watch what the same move does here, on randomized data.

In [ ]:
# C() tells the regression to treat vote history as categories (below / average / above),
# not as numbers. We add it as a control, exactly like adding a column in Week 2.
smf.ols('voted_2014 ~ treatment + C(vote_history_stratum)', data=df).fit().params['treatment']

Compare the two weeks **in the same units**, so the contrast is fair:

- **Week 2 (observational):** adding a control moved the program coefficient 0.087 → 0.034, a drop of about **0.053, or −61%** (more than half).
- **Week 3 (randomized):** adding a control moved the ATE 0.0073 → 0.0063, a drop of about **0.001, or −14%** (a tenth of a percentage point).

So controlling does *much less* here, but not *nothing*. Why does it do anything at all, if randomization balanced the groups? It *did* balance them: the treatment and control groups have nearly identical vote-history mixes. So the small move that remains is mostly **noise, not confounding** being removed.

There is also a second reason researchers control for a strong predictor like vote history even in a randomized experiment: it makes the estimate **more precise** (it tightens the uncertainty around the number). We will learn to read that precision in Week 5. In short:

In observational data, controlling fights confounding and still loses to what you can't measure. In a randomized experiment the confounding is already gone, so controlling barely moves the estimate and its real job is precision.

## Part 2: Could +0.7 have happened by chance? (Randomization inference)

We have an observed gap of about 0.7 percentage points. Is that big enough to be confident the mailer actually did something? Or is +0.7 within the range of gaps we'd see just from the coin flip, even if the mailer did nothing at all?

We answer this with **randomization inference**, which the framework slides defined as: re-randomize who gets treated, recompute the gap, repeat, and see how our actual gap compares to the fake ones.

Let's build this up one step at a time.

**A note on difficulty.** The code in this section is a step up from Week 2. The *idea* is simple: re-randomize, recompute, see whether our gap is unusual. The *code* introduces a `for` loop and a new pandas trick (`.sample()`). I found loops hard to get comfortable with. If you can follow the idea but the code feels opaque, that is normal at this stage. Focus on the logic first. The syntax becomes familiar through repetition, and you will see this same pattern in Weeks 4 and 5.

### Step 1: One fake re-assignment, by hand

If the mailer did nothing, then every voter would have voted exactly the way she did no matter which group the coin flip put her in. So the assignment we actually got is only one of the many the coin flip could have produced. Let's re-flip the coin once: assign treatment again at random, keeping the same number in each group, and recompute the difference of means.

This is ONE iteration of the randomization inference procedure. It produces ONE number: the gap we'd see under ONE fake re-assignment.

This is the "One Shuffle" slide, run on all 107,550 voters.

In [ ]:
# One fake re-assignment.
np.random.seed(2026)

# The key move: shuffle the treatment assignments.
# .sample(frac=1) randomly reorders every row of the column.
# .values pulls out just the numbers (0s and 1s), detached from their original rows.
# Result: each voter keeps her real outcome (voted_2014), but gets a NEW
# assignment, drawn at random.
shuffled_treatment = df['treatment'].sample(frac=1).values

# Compute the fake ATE: among voters assigned "treated" this time,
# what was their average turnout? Same for "control." Take the difference.
fake_treated_turnout = df['voted_2014'][shuffled_treatment == 1].mean()
fake_control_turnout = df['voted_2014'][shuffled_treatment == 0].mean()
fake_ate = fake_treated_turnout - fake_control_turnout

print(f'One fake re-assignment produced a gap of: {100*fake_ate:+.4f} percentage points')
print(f'(Our actual observed gap was:             {100*observed_ate:+.4f} percentage points)')

### Step 2: Now do that 1,000 times

One fake gap tells us almost nothing. We need to see the *distribution* of fake gaps: the range of values that chance alone would produce if there were no treatment effect. So we wrap the same three lines in a loop and store each result.

In [ ]:
# What this loop does in plain English:
# 1. Shuffle the treatment assignments (reassign who is "treated" at random)
# 2. Compute the difference in means under this fake assignment
# 3. Store that fake ATE
# 4. Repeat 1,000 times
# The result is a distribution of fake ATEs: what the gap would look like
# if the mailer did nothing.

np.random.seed(2026)  # reset so results are reproducible

n_reps = 1000
fake_ates = np.empty(n_reps)

for i in range(n_reps):
    # Shuffle the treatment assignments, same .sample(frac=1) move as Step 1.
    shuffled_treatment = df['treatment'].sample(frac=1).values
    # Compute the fake ATE under this new assignment.
    fake_ates[i] = (df['voted_2014'][shuffled_treatment == 1].mean()
                    - df['voted_2014'][shuffled_treatment == 0].mean())

# Convert to percentage points for easier reading.
fake_ates_pp = 100 * fake_ates
observed_ate_pp = 100 * observed_ate

# How many fake re-assignments produced a gap as extreme as our real one?
n_as_extreme = int(np.sum(np.abs(fake_ates_pp) >= abs(observed_ate_pp)))
p_value = n_as_extreme / n_reps

print(f'Observed ATE:                                    {observed_ate_pp:+.3f} pp')
print(f'Fake (null) ATE mean:                            {fake_ates_pp.mean():+.4f} pp  (close to zero, as expected)')
print(f'Fake (null) ATE range:                          [{fake_ates_pp.min():+.3f}, {fake_ates_pp.max():+.3f}] pp')
print()
print(f'Number of fake assignments with |ATE| >= ours:   {n_as_extreme} out of {n_reps}')
print(f'Two-sided p-value:                               {p_value:.4f}')

**Read that p-value out loud.** Out of 1,000 fake randomizations of the very same voters and outcomes, only 8 produced a gap as big as the one we actually observed. That is 8 in 1,000.

In plain English: *if the mailer really did nothing, you would see a gap of +0.73 percentage points (or larger) by chance less than 1 percent of the time*. The mailer almost certainly did something. The effect is small, and it is real.

That p-value of 0.008 is well under the 0.05 line the framework slides called "statistically significant." Remember what that does and does not buy you: it says chance is a poor explanation for the gap. It says nothing about whether a 0.7-point effect is worth \$143 a vote.

You can also see this by plotting the distribution of fake ATEs and marking the observed value:

In [ ]:
# The same picture as the slide: gray for the fake experiments that came out
# smaller than ours, gold for the ones that came out as big or bigger.
import matplotlib.pyplot as plt

bins = np.linspace(-1.0, 1.0, 41)
as_extreme  = fake_ates_pp[np.abs(fake_ates_pp) >= abs(observed_ate_pp)]
not_extreme = fake_ates_pp[np.abs(fake_ates_pp) < abs(observed_ate_pp)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(not_extreme, bins=bins, color='lightgray', edgecolor='gray')
ax.hist(as_extreme, bins=bins, color='goldenrod', edgecolor='gray')
ax.axvline(observed_ate_pp, color='navy', linewidth=2,
           label=f'Observed ATE = {observed_ate_pp:+.2f} pp')
# The mirror line. The test is two-sided, so a fake gap of -0.73 counts
# as just as extreme as our +0.73.
ax.axvline(-observed_ate_pp, color='navy', linewidth=2, linestyle='--')
ax.set_xlabel('ATE (percentage points)')
ax.set_ylabel('Number of fake randomizations')
ax.set_title(f'{n_reps} fake re-assignments of the same voters\n'
             f'{n_as_extreme} came out as big as ours (p = {p_value:.4f})')
ax.legend()
plt.tight_layout()
plt.show()

**Why this matters for the rest of the course.** Randomization inference is conceptually simple: it does in code exactly what the original randomization did in real life. There is no distribution to memorize, no formula to derive, no assumption that the data are normally distributed. The technique works because the experiment was randomized in the first place.

You will see versions of this idea in every later week. When the data set is small, or when the outcome is weird, or when the SE formula requires assumptions you can't justify, the answer is almost always: re-randomize in code and see what chance alone produces.

## Part 3: Where the effect is bigger

Look at the ATE separately by `vote_history_stratum`.

In [ ]:
table_vh = (df
            .groupby(['vote_history_stratum','treatment'])['voted_2014']
            .mean()
            .unstack('treatment')
            .rename(columns={0:'control', 1:'treated'}))
table_vh['ATE_pp'] = 100 * (table_vh['treated'] - table_vh['control'])
table_vh.round(4)

Read the `ATE_pp` column. The mailer looks like it is doing more work in the **below-median** stratum (people with low recent turnout, more room to move), and for above-median voters, people who were already voting in most elections, there is very little signal and even a noisy negative.

**Careful here, twice over.**

First, none of these three numbers is far enough from the others for us to say the effect really differs by stratum. Split a sample three ways and every subgroup gets noisier. In the full 1.97 million-voter paper the below-median and average groups are statistically indistinguishable, and the ordering is not even the same as ours: the paper reports **+0.8 pp for average, +0.7 for below, +0.4 for above, all three positive and all three significant**. Our above-median number came out negative. That is our subset being small, not the mailer backfiring on reliable voters.

Second, and this is the deeper point: the mailer was randomized, but **vote history was not**. We did not assign anyone to be a low-propensity voter. So a comparison *across* these rows is an observational comparison of the kind Week 2 was about, even though each row on its own is a clean experimental estimate. The paper says this about its own version of this table: the subgroup contrasts "are observational quantities."

Next week is about exactly this temptation.

Now split by `high_salience_state`. This is the paper's own measure: it scores each state 1 to 4 on how much was at stake in 2014 (a contested Senate race, a contested governor's race, and a Cook Political Report toss-up rating for each), and states scoring 3 or 4 count as high-salience. The 2017 paper's central question was whether the social pressure effect generalizes from low-salience contexts to high-salience ones.

In [ ]:
table_sal = (df
             .groupby(['high_salience_state','treatment'])['voted_2014']
             .mean()
             .unstack('treatment')
             .rename(columns={0:'control', 1:'treated'}))
table_sal['ATE_pp'] = 100 * (table_sal['treated'] - table_sal['control'])
table_sal.round(4)

Compare the two rows. **The effect is larger in low-salience states than in high-salience states**, about +1.3 pp against +0.3 pp. Two cautions. Our subset is too small to call that gap decisive on its own (p = 0.09), and it exaggerates: on all 1.97 million voters the paper finds the low-salience effect about **twice** the high-salience one, not four times. And the same warning as the table above applies, because salience is a feature of the state, not something anybody randomized. What the paper does say, in its abstract, is that the effect was largely consistent across the 17 states, with somewhat larger effects where election salience was lower. This is the empirical version of the claim from the framework slides: the same report card does different work in different settings. It is also the bridge to the disagreement with the 2008 paper.

The 2006 Michigan primary was extremely low-salience: a primary, in August, with control-group turnout of 29.7%. It is the leftmost edge of the salience scale. The +1.3-point effect in our low-salience-state cut is closer to the +4.8-point 2008 number than the +0.3-point effect in our high-salience cut is.

Internal validity does not tell you what to do at scale. Both of these numbers are valid. To predict what would happen if the donor's nonprofit mailed 20 million voters in 2026, you have to make an *external-validity* argument about which of these settings the 2026 cycle will look more like.

---

## What you've seen today

- `df.groupby(col)[outcome].mean()` is the entire causal computation for an experiment, *when* the treatment was assigned by a coin flip.
- **Randomization inference**: re-randomize the treatment assignments a thousand times, recompute the ATE each time, and ask whether the value we actually observed is unusual in that null distribution. No formula. No normal-distribution assumption. Just the same coin flip, simulated.
- An ATE is an average. It can hide meaningful heterogeneity by subgroup *and* by setting. The 2008/2014 disagreement is the most cleanly documented setting-effect example in the field.

Next, open `wk03_problem_set.ipynb`.